In [2]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(".."))
from src.config import SCHEMA_COLUMNS, LAT_MIN, LAT_MAX, LON_MIN, LON_MAX, OPERATOR_MAP

RAW_PATH = os.path.join("..", "data", "raw", "470.csv")
PROCESSED_PATH = os.path.join("..", "data", "processed", "cleaned_bd_cells.parquet")

print(f"Reading raw data from: {RAW_PATH}")
df = pd.read_csv(RAW_PATH, names=SCHEMA_COLUMNS, low_memory=False)
row_count = len(df)
print(f"Raw row ingested: {row_count:,}")


df = df[
    (df["lat"]>=LAT_MIN) & (df["lat"]<=LAT_MAX) &
    (df["lon"]>=LON_MIN) & (df["lon"]<=LON_MAX)
].copy()

df["operator"] = df["net"].map(OPERATOR_MAP)

df = df.dropna(subset=["operator", "lat", "lon"]).copy()


df["first_seen"] = pd.to_datetime(df["created"], unit="s", errors= "coerce")
df["last_seen"] = pd.to_datetime(df["updated"], unit="s", errors= "coerce")

df["averageSignal"] = df["averageSignal"].replace(0, np.nan)

df["radio"] = df["radio"].astype("category")
df["operator"] = df["operator"].astype("category")
df["mcc"] = df["mcc"].astype("int16")
df["net"] = df["net"].astype("int8")
df["cel"] = df["cell"].astype("int64")

cleaned_count = len(df)
print(f"Cleaned valid rocords: {cleaned_count:,}")

df.to_parquet(PROCESSED_PATH, index= False, engine = "pyarrow")
print(f"Successfully saved to: {PROCESSED_PATH}")

Reading raw data from: ../data/raw/470.csv
Raw row ingested: 8,945
Cleaned valid rocords: 8,945
Successfully saved to: ../data/processed/cleaned_bd_cells.parquet
